In [121]:
# math librairies
import numpy as np      # see: https://mathesaurus.sourceforge.net/matlab-numpy.html

from plotly import express as px
from plotly import subplots as splt
#from plotly import figure_factory as ff
import pandas               # used with plotly

# dnn
import torch
from torch.nn import MSELoss, Conv2d, ModuleList, Hardtanh
from torch.optim import Adam

# custom librairies 
import imicpe
import imicpe.optim as optim
print(imicpe.__version__)

1.0.15


## Choix de l'unité de calcul (CPU/GPU)

La fonction `chooseDevice()` fournie permet de choisir automatiquement le GPU s'il est disponible.  
Il est aussi possible de lui passer spécifiquement votre choix si besoin : 
- CPU : `device='cpu'` 
- GPU NVIDIA ou AMD : `device='cuda'`
- Processeur Apple M# : `device='mps'`

*Remarque.* Une bonne pratique consiste à **indiquer expressément l'unité de calcul** lors de l'utilisation des fonction de PyTorch quand c'est possible (en utilisant l'option `device=device`), par exemple lors de la création du dataset, etc.

In [122]:
device = optim.chooseDevice()

Using device: cpu



# **Initialisation**

## Chargement des bases de données (images de vérité terrain)

On utilise la base de donnée BSDS500 ([Berkeley Segmentation Data Set and Benchmarks 500](https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/grouping/resources.html)), qui contient 500 images, réparties en 
- 200 images pour l'entrainement,
- 100 images pour la validation,
- 200 images pour le test.

La classe `BSDSDataset` fournie permet de charger les images et de les stocker dans un objet qui agit comme une liste : 
- pour un dataset `DS`, on récupère sa longueur avec `len(DS)`,
- on accède à un élément du dataset via l'opérateur crochets, par exemple `DS[9]`.

In [123]:
dataset_options = dict(
    root_dir = "BSDS500/images",        # chemin local vers la base de données
    image_size = (128, 128),            # taille souhaitée pour les images, qui sont rognée si nécessaire
    image_cnt = None,                   # si spécifié, nb d'images extraites dans la base 
    gray = True,
    device = device,
)

train_bsds = optim.BSDSDataset(mode="train", **dataset_options)
val_bsds   = optim.BSDSDataset(mode="val",   **dataset_options)
test_bsds  = optim.BSDSDataset(mode="test",  **dataset_options)

print("Nombre d'images de la base d'entrainement :",  len(train_bsds))
print("Nombre d'images de la base de validation :",   len(val_bsds))
print("Nombre d'images de la base de test :",         len(test_bsds))


Nombre d'images de la base d'entrainement : 200
Nombre d'images de la base de validation : 100
Nombre d'images de la base de test : 200


In [124]:
# exemple
Nex = 9
xbar_ex, image_id = train_bsds[Nex]

fig = px.imshow(
            optim.torchImg2Numpy(xbar_ex), 
            range_color=[0,1], 
            color_continuous_scale='gray', 
            title="Exemple d'une image xbar de la base d'entrainement"
        )
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()

## Génération de la base d'images dégradées

### Implémentation du modèle de dégradation (modèle direct)

À partir de ces images de vérité terrain, on construit les données dégradées en ajoutant à chaque canal de l'image un bruit gaussien (voir [`torch.randn`](https://pytorch.org/docs/stable/generated/torch.randn.html)).

In [125]:
def addGaussianNoise(clean_image, sigma, seed=2024):        
    noise = torch.randn(clean_image.shape, device=clean_image.device, dtype=clean_image.dtype, generator=torch.Generator().manual_seed(seed)) * sigma
    noisy_image = clean_image + noise

    return noisy_image

### Génération des bases de données bruitées

La classe `NoisyDataset` permet de générer une base de données dégradée à partir d'une base d'images de vérité terrain, et accepte les arguments suivants :
- `dataset` : le dataset d'images de vérité terrain,
- `degradation_model` : la fonction modélisant la dégradation,
- `sigma` : l'écart-type du bruit gaussien,
- `seed` (facultatif) : la graine du générateur de nombre aléatoires.

De plus, la méthode `.toTensorDataset()` permet de stocker le dataset en mémoire si on le souhaite.

In [126]:
sigma = 0.1
seed = 2025         # for reproducibility

train_noisybsds = optim.NoisyDataset(train_bsds, degradation_model=addGaussianNoise, sigma=sigma, seed=seed).toTensorDataset()
val_noisybsds   = optim.NoisyDataset(val_bsds, degradation_model=addGaussianNoise, sigma=sigma, seed=seed).toTensorDataset()
test_noisybsds  = optim.NoisyDataset(test_bsds, degradation_model=addGaussianNoise, sigma=sigma, seed=seed).toTensorDataset()

print("Nombre d'images bruitées de la base d'entrainement :",  len(train_noisybsds))
print("Nombre d'images bruitées de la base de validation :",   len(val_noisybsds))
print("Nombre d'images bruitées de la base de test :",         len(test_noisybsds))


Nombre d'images bruitées de la base d'entrainement : 200
Nombre d'images bruitées de la base de validation : 100
Nombre d'images bruitées de la base de test : 200


In [127]:
# exemple (suite)
z_ex, xbar_ex, image_id = train_noisybsds[Nex]

img_xbar_ex = optim.torchImg2Numpy(xbar_ex)
img_z_ex    = optim.torchImg2Numpy(z_ex)

snr_ex = optim.snr(img_z_ex,img_xbar_ex)

fig = px.imshow(np.array([img_xbar_ex,img_z_ex]), color_continuous_scale='gray',
                    title="Exemple d'une paire (xbar,z) de la base d'entrainement",
                    facet_col=0, facet_col_spacing=0, facet_col_wrap=2,
                    width=800, height=400,
                    )
fig.update_coloraxes(cmin=0, cmax=1)

legend = ['Vérité terrain xbar','Donnée bruitée z (PSNR='  +"{:.2f}".format(snr_ex)+'dB)']
item_map={f'{i}':key for i, key in enumerate(legend)}
fig.for_each_annotation(lambda a: a.update(text=item_map[a.text.split("=")[1]]))

fig.show()

# ISTA déroulé pour le débruitage d'image

In [128]:
# choix d'une graine pour initialiser le générateur
torch.manual_seed(5117915234194007355)

## Création du réseau

### Mise en place de l'architecture du réseau

On utilise des couches convolutionnelles, donc la syntaxe PyTorch est la suivante :  

`torch.nn.Conv2d(input_channel, output_channel, kernel_size, bias=False, padding=1, padding_mode='zeros')`

In [129]:
class UnfoldedISTA(torch.nn.Module):
    def __init__(self, channels=3, features=5, depth=3):
        """ UnfoldedISTA construction

        Parameters
        ----------
        channels: int
            Nombre de canaux dans les images
        features: int
            Nombre de noeuds par couche
        depth: int
            Nombre de couches du réseau
        """
        
        super().__init__()

        # construction des operateurs
        self.D  = Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, padding=1, bias=False)     # primal to dual operator
        self.Dt = Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, padding=1, bias=False)     # dual to primal operator

        #  construction des matrices de poids et des vecteurs de biais
        self.bias_layers   = ModuleList()
        self.weight_layers = ModuleList()
        for i in range(depth):
            self.bias_layers  .append(Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, padding=1))
            self.weight_layers.append(Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, padding=1))

        # fonction d'activation
        self.activation = torch.nn.ReLU()

    def forward(self, z):

        u = torch.zeros_like(z)

        for k in range(len(self.bias_layers)):
            bias   = self.bias_layers[k]
            weight = self.weight_layers[k]

            u = self.activation(
                u - self.Dt(self.D(u) - z) + bias(u) + weight(u)
            )

        # dernière mise à jour sans activation
        bias   = self.bias_layers[-1]
        weight = self.weight_layers[-1]

        x = u - self.Dt(self.D(u) - z) + bias(u) + weight(u)

        return x



### Instanciation du réseau

In [130]:
# paramètres du réseau
Nchannels = 1               # nb de canaux dans les images
Nnodes    = 16               # nb de noeuds par couche
Nlayers   = 5               # nb de couches

# création du réseau à partir des paramètres
ista_net = UnfoldedISTA(channels=Nchannels, features=Nnodes, depth=Nlayers).to(device)

print("Nombre de paramètres :", sum(w.numel() for w in ista_net.parameters()))

Nombre de paramètres : 118


## Entrainement

### Configuration de l'entrainement

La classe `Trainer` fournie permet d'encapsuler le processus d'entrainement du réseau, et nécessite les arguments suivants :
- `model`: le réseau à entrainer,
- `optimizer`: l'algorithme d'optimisation utilisé pour minimiser la loss,
- `loss`: la fonction de loss à minimiser,
- `train_dataset`: la base donnée d'entrainement,
- `val_dataset` (optionnel): la base donnée de validation,
- `check_val_every_n_epoch` (optionnel): si le dataset de validation est fourni, cela permet de réaliser une étape de validation toutes les $n$ epochs d'entrainement (cela permet de réduire le temps de calcul).

L'entrainement nécessite de plus de régler les 3 paramètres suivants :
- la taille `batch_size` d'un batch (sous-ensemble du dataset), qui correspond au nombre d'images utilisées avant de mettre à jour les paramètres du modèle,
- le nombre d'epochs `Nepochs`, qui correspond au nombre de fois que l'algorithme d'apprentissage voit l'intégralité des données du dataset d'entrainement,
- le nombre `NepochsToValidate` d'epochs d'entrainement à réaliser avant d'effectuer une étape de validation.

Pour schématiser, on peut voir l'entrainement comme une boucle for sur le nombre d'epochs, où chaque tour voit l'ensemble des données d'entrainement. Dans cette boucle est imbriquée une autre boucle, qui itère sur chaque batch.

In [131]:
# paramètres
Nepochs    = 100             # nb d'epochs
batch_size = 16              # taille d'un batch
NepochsToValidate = 10       # fréquence de validation

# choix de la loss
loss      = torch.nn.L1Loss()
optimizer = Adam(ista_net.parameters(), lr=1e-3)

# initialisation de l'entrainement
ista_trainer = optim.Trainer(ista_net, optimizer, loss, batch_size,
                  train_noisybsds, val_noisybsds, check_val_every_n_epoch=NepochsToValidate)


In [132]:
ista_trainer.run(Nepochs)

Training:   0%|          | 0/100 [00:00<?, ?epoch/s]

Validation epoch   10 | Loss: 1.29e-01 | PSNR: 1.53e+01 | Wall time: 3.01e-01
Validation epoch   20 | Loss: 6.83e-02 | PSNR: 2.07e+01 | Wall time: 2.47e-01
Validation epoch   30 | Loss: 5.61e-02 | PSNR: 2.24e+01 | Wall time: 2.97e-01
Validation epoch   40 | Loss: 5.17e-02 | PSNR: 2.31e+01 | Wall time: 2.55e-01
Validation epoch   50 | Loss: 5.01e-02 | PSNR: 2.35e+01 | Wall time: 2.80e-01
Validation epoch   60 | Loss: 4.75e-02 | PSNR: 2.40e+01 | Wall time: 2.59e-01
Validation epoch   70 | Loss: 4.79e-02 | PSNR: 2.41e+01 | Wall time: 4.51e-01
Validation epoch   80 | Loss: 4.56e-02 | PSNR: 2.45e+01 | Wall time: 2.83e-01
Validation epoch   90 | Loss: 4.54e-02 | PSNR: 2.46e+01 | Wall time: 2.75e-01
Validation epoch  100 | Loss: 4.50e-02 | PSNR: 2.47e+01 | Wall time: 2.55e-01


### Affichage des performances d'entrainement et validation

In [133]:
data_train = optim.getData(ista_trainer, 'Training')
data_valid = optim.getData(ista_trainer, 'Validation')        

plt_train_loss = pandas.DataFrame({'x':data_train["epoch"], 'y':data_train["Loss"], 'legend':'training loss',   'type':'loss'})
plt_valid_loss = pandas.DataFrame({'x':data_valid["epoch"], 'y':data_valid["Loss"], 'legend':'validation loss', 'type':'loss'})
plt_loss       = pandas.concat([plt_train_loss,plt_valid_loss]) 
    
figLoss = px.line(plt_loss,
            x='x', 
            y='y', 
            log_x=False, log_y=True,
            color='legend',
            labels={'x':'epochs','y':'Loss (logscale)'},
            title='Évolution de la fonction de coût en fonction des epochs',
            width=800, height=450)
figLoss.show()


plt_train_psnr = pandas.DataFrame({'x':data_train["epoch"], 'y':data_train["PSNR"], 'legend':'training PSNR',   'type':'PSNR'})
plt_valid_psnr = pandas.DataFrame({'x':data_valid["epoch"], 'y':data_valid["PSNR"], 'legend':'validation PSNR', 'type':'PSNR'})
plt_psnr       = pandas.concat([plt_train_psnr,plt_valid_psnr]) 
    
figPSNR = px.line(plt_psnr,
            x='x', 
            y='y', 
            log_x=False, log_y=True,
            color='legend',
            labels={'x':'epochs','y':'PSNR (logscale)'},
            title='Évolution du PSNR en fonction des epochs',
            width=800, height=450)
figPSNR.show()

## Test

In [134]:
# chargement d'une image de test
z, xbar, img_id = test_noisybsds[0]

img_xbar = optim.torchImg2Numpy(xbar)
img_z    = optim.torchImg2Numpy(z)

snr_z = optim.snr(img_z,img_xbar)

# application du réseau à la donnée dégradée z
with torch.no_grad():
    xhat = ista_net(z.unsqueeze(0).to(device)).squeeze(0).cpu()


img_xhat = optim.torchImg2Numpy(xhat)  
snr_xhat = optim.snr(img_xhat,img_xbar) 
    
# affichage du résultat

fig = px.imshow(np.array([img_xbar,img_z,img_xhat]), color_continuous_scale='gray',
                    title='id image : '+ str(img_id),
                    facet_col=0, facet_col_spacing=0, facet_col_wrap=3,
                    width=800, height=400,
                    )
    
fig.update_coloraxes(cmin=0, cmax=1)

legend = [  'Vérité terrain xbar',
            'Donnée bruitée z (PSNR='  +"{:.2f}".format(snr_z)+'dB)',
            'Estimation xhat (PSNR='+"{:.2f}".format(snr_xhat)+'dB)'   ]
item_map={f'{i}':key for i, key in enumerate(legend)}
fig.for_each_annotation(lambda a: a.update(text=item_map[a.text.split("=")[1]]))

fig.show()